# Efficient Batch Inference for Text Attribution

This notebook follows `efficient-batch-inference-text.py` step by step and shows how to run perturbation-based attribution for pure LLM prompts.

In [ ]:
import os

# Run notebook cells from the repository root, matching the other tutorials.
if os.path.basename(os.getcwd()) == "tutorial":
    os.chdir("../")

os.getcwd()

## Imports and Cache Setup

In [ ]:
import csv
import json
from pathlib import Path

import numpy as np
from IPython.display import HTML, SVG, display

from text_attribution import (
    EfficientLLMSubModularExplanationText,
    QwenTextAdaptor,
    build_text_span_masks,
)
from text_attribution.output_token_input_influence import (
    compute_output_token_input_influence,
    save_output_token_input_influence_csv,
    save_output_token_input_influence_html,
    save_output_token_input_influence_json,
    save_output_token_input_influence_svg,
)
from visualization.combined_text_attribution_report import save_combined_text_attribution_report
from visualization.text_attribution_visualization import (
    save_text_attribution_html,
    save_text_saliency_svg,
)

# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com" # for Chinese users
os.environ.setdefault("HF_HOME", "./model_checkpoint/hf_cache")

In [ ]:
def tensor_to_list(value):
    if hasattr(value, "detach"):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value

## Configuration

These variables mirror the command-line arguments in `efficient-batch-inference-text.py`.

In [ ]:
model_name = "Qwen/Qwen3-8B"
system_prompt = "You are a precise assistant. Answer with a concise explanation."
user_prompt = "Explain why batch inference can make perturbation-based attribution faster."

max_new_tokens = 96
batch_size = None
search_scope = 8
pending_samples = 4
update_step = 10
lambda1 = 1.0
lambda2 = 1.0
mask_strategy = "replace"  # choices: "replace", "attention"
input_granularity = "sentence"  # choices: "sentence", "message", "readable"
target_token_limit = None  # set to an integer to explain only the first N generated tokens

output_dir = Path("./text_attribution_outputs/notebook_text_attribution")
output_dir.mkdir(parents=True, exist_ok=True)

## Build the Chat Prompt

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

messages

## Load the Qwen Text Adaptor

In [ ]:
qwen = QwenTextAdaptor.from_pretrained(model_name)

## Convert Text into Attribution Regions

`build_text_span_masks` renders the chat template, tokenizes it, and creates the candidate text regions that can be kept or masked during attribution.

In [ ]:
rendered_text, input_ids, V_set, spans = build_text_span_masks(
    qwen.tokenizer,
    messages,
    enable_thinking=False,
    granularity=input_granularity,
)

print(rendered_text)
print(f"Input token shape: {tuple(input_ids.shape)}")
print(f"Number of attribution regions: {len(spans)}")

In [ ]:
for idx, span in enumerate(spans):
    print(idx, span.to_dict())

## Generate the Answer and Select Target Tokens

In [ ]:
generation = qwen.generate_from_rendered_chat(
    rendered_text,
    max_new_tokens=max_new_tokens,
    do_sample=False,
)

selected_token_indices = list(range(len(generation["generated_answer_ids"])))
if target_token_limit is not None:
    selected_token_indices = selected_token_indices[:target_token_limit]
if not selected_token_indices:
    raise ValueError("No generated tokens were selected for attribution.")

qwen.set_targets(generation["generated_answer_ids"], selected_token_indices)

output_tokens = generation["output_tokens"]
selected_tokens = [output_tokens[index] for index in selected_token_indices]

print(generation["output_text"])
print(f"Generated tokens: {len(output_tokens)}")
print(f"Selected tokens: {selected_tokens}")

## Run Efficient Batch Text Attribution

In [ ]:
explainer = EfficientLLMSubModularExplanationText(
    qwen,
    lambda1=lambda1,
    lambda2=lambda2,
    search_scope=search_scope,
    pending_samples=pending_samples,
    update_step=update_step,
    batch_size=batch_size,
    mask_strategy=mask_strategy,
)

S_set, saved_json_file = explainer(input_ids, V_set)

## Package the Attribution Result

In [ ]:
saved_json_file.update(
    {
        "model_name": model_name,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "rendered_prompt": rendered_text,
        "input_granularity": input_granularity,
        "input_ids": tensor_to_list(input_ids),
        "spans": [span.to_dict() for span in spans],
        "ordered_masks": tensor_to_list(S_set),
        "generated_answer_ids": tensor_to_list(generation["generated_answer_ids"]),
        "output_text": generation["output_text"],
        "output_tokens": output_tokens,
        "selected_interpretation_token_id": selected_token_indices,
        "selected_interpretation_token_word_id": tensor_to_list(
            qwen.selected_interpretation_token_word_id
        ),
        "selected_interpretation_tokens": selected_tokens,
    }
)

## Save JSON and Insertion/Deletion Scores

In [ ]:
json_path = output_dir / "text_attribution_result.json"
json_path.write_text(
    json.dumps(saved_json_file, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

csv_path = output_dir / "insertion_deletion_scores.csv"
with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["step", "region_area", "insertion_score", "deletion_score", "smdl_score"])
    rows = zip(
        saved_json_file.get("region_area", []),
        saved_json_file.get("insertion_score", []),
        saved_json_file.get("deletion_score", []),
        saved_json_file.get("smdl_score", []),
    )
    for step, (region_area, insertion_score, deletion_score, smdl_score) in enumerate(rows):
        writer.writerow([step, region_area, insertion_score, deletion_score, smdl_score])

print(f"Saved attribution JSON to {json_path}")
print(f"Saved insertion/deletion CSV to {csv_path}")

## Save Text Attribution Visualizations

In [ ]:
html_path = output_dir / "text_attribution_visualization.html"
save_text_attribution_html(
    saved_json_file=saved_json_file,
    spans=spans,
    output_tokens=output_tokens,
    output_text=generation["output_text"],
    model_name=model_name,
    save_path=html_path,
)

saliency_svg_path = output_dir / "text_saliency_map.svg"
save_text_saliency_svg(
    saved_json_file=saved_json_file,
    spans=spans,
    save_path=saliency_svg_path,
)

print(f"Saved visualization HTML to {html_path}")
print(f"Saved saliency map SVG to {saliency_svg_path}")

## Save Output-Token/Input-Region Influence Views

In [ ]:
token_input_influence = compute_output_token_input_influence(saved_json_file)

token_input_influence_json_path = output_dir / "output_token_input_influence.json"
token_input_influence_csv_path = output_dir / "output_token_input_influence.csv"
token_input_influence_html_path = output_dir / "output_token_input_influence.html"
token_input_influence_svg_path = output_dir / "output_token_input_influence.svg"

save_output_token_input_influence_json(token_input_influence, token_input_influence_json_path)
save_output_token_input_influence_csv(token_input_influence, token_input_influence_csv_path)
save_output_token_input_influence_html(token_input_influence, token_input_influence_html_path)
save_output_token_input_influence_svg(token_input_influence, token_input_influence_svg_path)

print(f"Saved output token input influence JSON to {token_input_influence_json_path}")
print(f"Saved output token input influence CSV to {token_input_influence_csv_path}")
print(f"Saved output token input influence HTML to {token_input_influence_html_path}")
print(f"Saved output token input influence SVG to {token_input_influence_svg_path}")

## Save and Display the Combined Report

In [ ]:
combined_report_path = output_dir / "combined_attribution_report.svg"
save_combined_text_attribution_report(
    saved_json_file=saved_json_file,
    token_input_influence=token_input_influence,
    save_path=combined_report_path,
)

print(f"Saved combined attribution report SVG to {combined_report_path}")
display(SVG(filename=str(combined_report_path)))

## Example Combined Attribution Report

The repository also includes a ready-made report at `examples/combined_attribution_report.svg`.

In [ ]:
display(SVG(filename="examples/combined_attribution_report.svg"))